# Generacion del Dataset Sintetico de Demanda ALDIMI

Este notebook genera:
 `aldimi_demand_dataset.csv` - Dataset de demanda diaria de medicamentos

**Contexto**: ALDIMI atiende ~100 familias con pacientes oncologicos. Cada paciente tiene un nivel de riesgo (Low/Medium/High) que determina que medicamentos necesita.

In [35]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)

# Rutas
BASE_DIR = Path(".").resolve().parent  # raiz del proyecto
CANCER_DATA = BASE_DIR / "cancer patient data sets.xlsx"
SALES_DATA = BASE_DIR / "DATASETS" / "pharmasalesdata" / "salesdaily.csv"
RAW_DIR = BASE_DIR / "datos" / "datos_modelo1" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Rutas configuradas:")
print(f"  Cancer data: {CANCER_DATA}")
print(f"  Sales data:  {SALES_DATA}")
print(f"  Output dir:  {RAW_DIR}")

Rutas configuradas:
  Cancer data: C:\Users\USER\Documents\2026-1\Machine Learning\ML3037-GRUPO4-ALDIMI\cancer patient data sets.xlsx
  Sales data:  C:\Users\USER\Documents\2026-1\Machine Learning\ML3037-GRUPO4-ALDIMI\DATASETS\pharmasalesdata\salesdaily.csv
  Output dir:  C:\Users\USER\Documents\2026-1\Machine Learning\ML3037-GRUPO4-ALDIMI\datos\datos_modelo1\raw


## 1. Analisis del dataset de pacientes por nivel de riesgo

Agrupamos los pacientes del dataset de cancer por su nivel (`Level`) para entender la distribucion real y luego forzar la proporcion deseada: **60 Low / 25 Medium / 15 High**.

In [36]:
cancer_df = pd.read_excel(CANCER_DATA)
cancer_df = cancer_df.drop("Patient Id", axis=1)

# Group by Level
level_counts = cancer_df["Level"].value_counts()
print("Distribucion real del dataset de cancer:")
print(level_counts)
print(f"\nTotal pacientes: {len(cancer_df)}")
print(f"Proporciones reales: Low={level_counts.get('Low',0)/len(cancer_df)*100:.1f}%, "
      f"Medium={level_counts.get('Medium',0)/len(cancer_df)*100:.1f}%, "
      f"High={level_counts.get('High',0)/len(cancer_df)*100:.1f}%")

Distribucion real del dataset de cancer:
Level
High      365
Medium    332
Low       303
Name: count, dtype: int64

Total pacientes: 1000
Proporciones reales: Low=30.3%, Medium=33.2%, High=36.5%


## 2. Definir distribucion objetivo para ALDIMI

Forzamos la distribucion a **60 Low / 25 Medium / 15 High** (total = 100 pacientes).

In [37]:
# Distribucion objetivo para ALDIMI
TARGET_LOW = 60
TARGET_MEDIUM = 25
TARGET_HIGH = 15
TARGET_TOTAL = TARGET_LOW + TARGET_MEDIUM + TARGET_HIGH

# Probabilidades de asignacion para nuevos ingresos
P_NEW_LOW = TARGET_LOW / TARGET_TOTAL      # 0.60
P_NEW_MEDIUM = TARGET_MEDIUM / TARGET_TOTAL  # 0.25
P_NEW_HIGH = TARGET_HIGH / TARGET_TOTAL      # 0.15

print(f"Distribucion objetivo: Low={TARGET_LOW}, Medium={TARGET_MEDIUM}, High={TARGET_HIGH}")
print(f"Proporciones para nuevos ingresos: Low={P_NEW_LOW:.0%}, Medium={P_NEW_MEDIUM:.0%}, High={P_NEW_HIGH:.0%}")

Distribucion objetivo: Low=60, Medium=25, High=15
Proporciones para nuevos ingresos: Low=60%, Medium=25%, High=15%


## 3. Parametros de simulacion

In [38]:
# Probabilidades diarias de transicion (balanceadas para equilibrio 60/25/15)
P_LOW_TO_MED = 0.0083   # Low -> Medium (empeora)
P_MED_TO_LOW = 0.020    # Medium -> Low (mejora)
P_MED_TO_HIGH = 0.012   # Medium -> High (empeora)
P_HIGH_TO_MED = 0.020   # High -> Medium (mejora)
P_DISCHARGE = 0.008     # Paciente dado de alta

# Dosis diarias por paciente (unidades/paciente/dia)
DOSE_N02BE = 2.0   # Paracetamol - todos los pacientes
DOSE_N05B = 0.5    # Ansioliticos - Medium + High
DOSE_M01AB = 1.0   # Diclofenaco - solo High

# Rango de poblacion
POP_MIN = 90
POP_MAX = 110
POP_TARGET = 100

print("Parametros definidos.")
print(f"  Dosis: N02BE={DOSE_N02BE}, N05B={DOSE_N05B}, M01AB={DOSE_M01AB}")
print(f"  Poblacion: {POP_MIN}-{POP_MAX} (target={POP_TARGET})")

Parametros definidos.
  Dosis: N02BE=2.0, N05B=0.5, M01AB=1.0
  Poblacion: 90-110 (target=100)


## 4. Cargar patrones temporales de salesdaily.csv

Usamos los patrones de estacionalidad de la farmacia como referencia para que la demanda no sea una linea plana.

In [39]:
sales_df = pd.read_csv(SALES_DATA)
sales_df["date"] = pd.to_datetime(sales_df["datum"], format="%m/%d/%Y", errors="coerce")
sales_df = sales_df.sort_values("date").dropna(subset=["date"])
sales_df = sales_df.drop_duplicates(subset=["date"], keep="first")
sales_df = sales_df.set_index("date").asfreq("D")

# Columnas ATC relevantes
ATC_COLS = ["N02BE", "N05B", "M01AB"]
for col in ATC_COLS:
    sales_df[col] = pd.to_numeric(sales_df[col], errors="coerce").fillna(0.0)

# Factores de variacion temporal (ratio vs media)
medias = {col: sales_df[col].mean() for col in ATC_COLS}
factores = {}
for col in ATC_COLS:
    if medias[col] > 0:
        factores[col] = (sales_df[col] / medias[col]).values
    else:
        factores[col] = np.ones(len(sales_df))

fechas = sales_df.index
n_dias = len(fechas)

print(f"Rango temporal: {fechas.min().date()} -> {fechas.max().date()}")
print(f"Total dias: {n_dias}")
print(f"Medias de referencia:")
for col in ATC_COLS:
    print(f"  {col}: {medias[col]:.2f}")

Rango temporal: 2014-01-02 -> 2019-10-08
Total dias: 2106
Medias de referencia:
  N02BE: 29.92
  N05B: 8.85
  M01AB: 5.03


## 5. Funciones de simulacion

In [40]:
def simulate_transitions(n_low, n_med, n_high):
    # Low -> Medium
    to_med = np.random.binomial(n_low, P_LOW_TO_MED)
    n_low -= to_med
    n_med += to_med

    # Medium -> Low (mejora)
    to_low = np.random.binomial(n_med, P_MED_TO_LOW)
    n_med -= to_low
    n_low += to_low

    # Medium -> High (empeora)
    to_high = np.random.binomial(n_med, P_MED_TO_HIGH)
    n_med -= to_high
    n_high += to_high

    # High -> Medium (mejora)
    to_med2 = np.random.binomial(n_high, P_HIGH_TO_MED)
    n_high -= to_med2
    n_med += to_med2

    return n_low, n_med, n_high


def assign_new_patient_level():
    r = np.random.random()
    if r < P_NEW_LOW:
        return "Low"
    elif r < P_NEW_LOW + P_NEW_MEDIUM:
        return "Medium"
    else:
        return "High"


def simulate_arrivals_departures(n_low, n_med, n_high):
    total = n_low + n_med + n_high

    # Altas (solo si total > 95)
    if total > 95:
        dp = P_DISCHARGE * (1 + (total - POP_TARGET) / 20)
        dp = min(dp, 0.05)
        d_low = np.random.binomial(n_low, dp * 1.5)
        d_med = np.random.binomial(n_med, dp * 0.5)
        d_high = np.random.binomial(n_high, dp * 0.2)
        n_low -= d_low
        n_med -= d_med
        n_high -= d_high

    total = n_low + n_med + n_high

    # Nuevos ingresos (solo si total < 105)
    if total < 105:
        expected = max(0.5, 1.0 + (POP_TARGET - total) / 15)
        n_arr = np.random.poisson(expected)
        n_arr = min(n_arr, POP_MAX - total)
        for _ in range(n_arr):
            level = assign_new_patient_level()
            if level == "Low":
                n_low += 1
            elif level == "Medium":
                n_med += 1
            else:
                n_high += 1

    return max(0, n_low), max(0, n_med), max(0, n_high)

print("Funciones de simulacion definidas.")

Funciones de simulacion definidas.


## 6. Ejecutar simulacion (2106 dias)

In [ ]:
n_low, n_med, n_high = TARGET_LOW, TARGET_MEDIUM, TARGET_HIGH

records = []

for day_idx in range(n_dias):
    fecha = fechas[day_idx]
    dia_semana = fecha.dayofweek  
    es_finde = int(dia_semana >= 5)

    n_low, n_med, n_high = simulate_transitions(n_low, n_med, n_high)

    n_low, n_med, n_high = simulate_arrivals_departures(n_low, n_med, n_high)

    total = n_low + n_med + n_high

    if es_finde:
        pct_low_sale = np.random.uniform(0.40, 0.60)
        pct_med_sale = np.random.uniform(0.10, 0.20)
        pres_low = round(n_low * (1 - pct_low_sale))
        pres_med = round(n_med * (1 - pct_med_sale))
        pres_high = n_high
    else:
        pres_low = n_low
        pres_med = n_med
        pres_high = n_high

    pres_total = pres_low + pres_med + pres_high

    pac_N02BE = pres_low + pres_med + pres_high   
    pac_N05B = pres_med + pres_high                
    pac_M01AB = pres_high                         

    dem_base_N02BE = pac_N02BE * DOSE_N02BE
    dem_base_N05B = pac_N05B * DOSE_N05B
    dem_base_M01AB = pac_M01AB * DOSE_M01AB

    f_n02be = factores["N02BE"][day_idx]
    f_n05b = factores["N05B"][day_idx]
    f_m01ab = factores["M01AB"][day_idx]

    dem_N02BE = max(0, dem_base_N02BE * f_n02be + np.random.normal(0, 2))
    dem_N05B = max(0, dem_base_N05B * f_n05b + np.random.normal(0, 1))
    dem_M01AB = max(0, dem_base_M01AB * f_m01ab + np.random.normal(0, 0.5))

    records.append({
        "fecha": fecha,
        "dia_semana": dia_semana,
        "es_fin_de_semana": es_finde,
        "n_total": total,
        "n_low": n_low,
        "n_medium": n_med,
        "n_high": n_high,
        "n_presentes": pres_total,
        "presentes_low": pres_low,
        "presentes_medium": pres_med,
        "presentes_high": pres_high,
        "N02BE_demand": round(dem_N02BE, 2),
        "N05B_demand": round(dem_N05B, 2),
        "M01AB_demand": round(dem_M01AB, 2),
    })

demand_df = pd.DataFrame(records)
demand_df["total_demand"] = demand_df["N02BE_demand"] + demand_df["N05B_demand"] + demand_df["M01AB_demand"]

print(f"Simulacion completada: {len(demand_df)} dias generados")

Simulacion completada: 2106 dias generados


## 7. Resumen del dataset generado

In [42]:
print("=== POBLACION ===")
print(f"  Media total: {demand_df['n_total'].mean():.1f}")
print(f"  Rango: {demand_df['n_total'].min()} - {demand_df['n_total'].max()}")
print(f"  Media Low: {demand_df['n_low'].mean():.1f}")
print(f"  Media Medium: {demand_df['n_medium'].mean():.1f}")
print(f"  Media High: {demand_df['n_high'].mean():.1f}")

print("\n=== DEMANDA DIARIA (unidades) ===")
for col in ["N02BE_demand", "N05B_demand", "M01AB_demand", "total_demand"]:
    print(f"  {col}: media={demand_df[col].mean():.1f}, "
          f"min={demand_df[col].min():.1f}, max={demand_df[col].max():.1f}")

finde = demand_df[demand_df["es_fin_de_semana"] == 1]
semana = demand_df[demand_df["es_fin_de_semana"] == 0]
print("\n=== EFECTO FIN DE SEMANA ===")
for col in ["N02BE_demand", "N05B_demand", "M01AB_demand"]:
    print(f"  {col}: semana={semana[col].mean():.1f}, finde={finde[col].mean():.1f}")

=== POBLACION ===
  Media total: 102.4
  Rango: 95 - 108
  Media Low: 50.5
  Media Medium: 29.3
  Media High: 22.6

=== DEMANDA DIARIA (unidades) ===
  N02BE_demand: media=186.1, min=0.0, max=1120.3
  N05B_demand: media=25.2, min=0.0, max=177.2
  M01AB_demand: media=22.8, min=0.0, max=96.3
  total_demand: media=234.2, min=0.0, max=1184.2

=== EFECTO FIN DE SEMANA ===
  N02BE_demand: semana=195.0, finde=163.8
  N05B_demand: semana=27.6, finde=19.2
  M01AB_demand: semana=21.9, finde=25.1


## 8. Guardar datasets

- `aldimi_demand_dataset.csv` -> dataset de demanda (va a `raw/` para hacer EDA y Feature Engineering)

In [ ]:
demand_cols = ["fecha", "dia_semana", "es_fin_de_semana",
               "n_total", "n_low", "n_medium", "n_high",
               "n_presentes",
               "N02BE_demand", "N05B_demand", "M01AB_demand", "total_demand"]
demand_path = RAW_DIR / "aldimi_demand_dataset.csv"
demand_df[demand_cols].to_csv(demand_path, index=False)
print(f"[OK] Guardado: {demand_path}")

print("\nDatasets generados.")

[OK] Guardado: C:\Users\USER\Documents\2026-1\Machine Learning\ML3037-GRUPO4-ALDIMI\datos\datos_modelo1\raw\aldimi_demand_dataset.csv

Datasets generados.
